<a href="https://colab.research.google.com/github/ThonyArga/analisis-lampara-hmc/blob/main/Analisis_degradacion_lampara_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de degradación de lámpara — versión final

**Decisiones metodológicas clave:**
- Modelo del área: seleccionado por AICc
- Modelo de V₆: **Seminormal** por criterio LOO RMSE (mejor capacidad predictiva empírica)
- Justificación: los modelos de 3 parámetros ajustan su asíntota exactamente al último dato observado, lo cual es sospechoso

**Contenido:**
- Fix del 199 V fantasma
- 8 modelos comparados con RMSE, R², R²aj, AICc y LOO
- Bootstrap del día de cruce
- Correlaciones múltiples con control por tiempo
- **Reporte HTML + DOCX con todas las tablas, gráficos y recomendaciones**


In [ ]:
!pip install -q openpyxl dcor python-docx

In [ ]:
from pathlib import Path
import shutil, itertools, base64

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.optimize import least_squares, brentq
from scipy.stats import norm, spearmanr, kendalltau, pearsonr, t as student_t

try:
    import dcor; DCOR_OK = True
except ImportError:
    DCOR_OK = False

try:
    from docx import Document
    from docx.shared import Inches, Pt
    from docx.enum.text import WD_ALIGN_PARAGRAPH
    DOCX_OK = True
except ImportError:
    DOCX_OK = False

try:
    from IPython.display import display, HTML
except ImportError:
    display, HTML = print, None

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plt.style.use("seaborn-v0_8-whitegrid")

# --- Formateo con coma decimal ---
comma_fmt = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ","))

def apply_comma(fig):
    for ax in fig.axes:
        ax.xaxis.set_major_formatter(comma_fmt)
        ax.yaxis.set_major_formatter(comma_fmt)

# --- Patch idempotente de plt.subplots ---
if not getattr(plt, "_comma_patched", False):
    _orig_subplots = plt.subplots
    def subplots_comma(*args, **kwargs):
        fig, ax = _orig_subplots(*args, **kwargs)
        apply_comma(fig)
        return fig, ax
    plt.subplots = subplots_comma
    plt._comma_patched = True

# --- Constantes ---
INITIAL_DATE           = pd.Timestamp("2026-05-23")
COMPLETE_BASELINE_DATE = pd.Timestamp("2026-05-30")
INSTALLATION_DATE      = pd.Timestamp("2026-05-19")
DAYS_OFFSET = (COMPLETE_BASELINE_DATE - INSTALLATION_DATE).days  # = 11

AREA_V_MIN, AREA_V_MAX = 160.0, 190.0
CERT_U_EXPANDED_REL, CERT_K = 0.022, 2.0
CERT_U_STANDARD_REL = CERT_U_EXPANDED_REL / CERT_K
RESULT_K = 2.0
TARGET_POWER_W, OPERATING_V_MAX = 6.0, 190.0

V6_MODEL_OVERRIDE = "Seminormal"

OUTPUT_DIR = Path("resultados_lampara")
OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figuras"
FIG_DIR.mkdir(exist_ok=True)

REPORT_ITEMS = []
def register_table(title, df, note=""):
    REPORT_ITEMS.append({"type":"table", "title":title, "df":df, "note":note})
def register_figure(title, path, caption=""):
    REPORT_ITEMS.append({"type":"figure", "title":title, "path":str(path), "caption":caption})
def register_text(title, body):
    REPORT_ITEMS.append({"type":"text", "title":title, "body":body})

print("Setup OK")
print(f"dcor: {DCOR_OK} | python-docx: {DOCX_OK}")
print(f"Modelo V6 forzado: {V6_MODEL_OVERRIDE}")
print(f"DAYS_OFFSET: {DAYS_OFFSET}")
print(f"apply_comma: {apply_comma}")

In [ ]:
import numpy, pandas, scipy, matplotlib, statsmodels, dcor, openpyxl, docx, sys
for m in [numpy, pandas, scipy, matplotlib, statsmodels, dcor, openpyxl, docx]:
    print(m.__name__, getattr(m, "__version__", "?"))
print("Python", sys.version)

In [ ]:
def show_colab_table(frame, caption):
    if HTML is None:
        print(caption); display(frame); return
    html_table = frame.to_html(index=False, escape=False, classes="rt", border=0)
    display(HTML(f'''
    <style>
    .rt {{border-collapse:collapse;margin:8px 0 20px 0;font-size:13px;}}
    .rt th {{background:#176B87;color:white;padding:6px 10px;text-align:center;}}
    .rt td {{padding:5px 10px;text-align:right;border-bottom:1px solid #D9E2E8;}}
    .rt tr:nth-child(even) {{background:#F4F8FA;}}
    </style>
    <div style="font-size:15px;font-weight:700;margin-top:12px;">{caption}</div>
    {html_table}'''))

## 1. Carga del Excel

In [1]:
from pathlib import Path
import pandas as pd

# 1. Definir las rutas de los posibles archivos
ruta_real = Path("Organized Data Sheet(1).xlsx") # Tu archivo privado
ruta_sintetica = Path("datos_sinteticos.xlsx")   # El archivo que generaste

# 2. Lógica de selección
if ruta_real.exists():
    data_file = ruta_real
    print("✅ Usando datos REALES (Privados).")
elif ruta_sintetica.exists():
    data_file = ruta_sintetica
    print("✅ Usando datos SINTÉTICOS (Públicos para GitHub).")
else:
    print("⚠️ No se encontró ningún archivo. Por favor, sube uno.")
    from google.colab import files
    up = files.upload()
    data_file = Path(list(up.keys())[0])

print("Archivo seleccionado:", data_file.resolve())
workbook = pd.ExcelFile(data_file)
print("Hojas encontradas:", workbook.sheet_names)

TypeError: 'NoneType' object is not subscriptable

## 2. Preparación de datos

In [ ]:
raw = pd.read_excel(data_file, sheet_name="Current Lamp Study Data")
raw.columns = [str(c).strip() for c in raw.columns]
reading_cols = [f"Measured Power {i} (W)" for i in range(1, 6)]
req = ["Date", "Voltage (V)", *reading_cols, "Average Power (W)", "Std Dev (W)"]
missing = [c for c in req if c not in raw.columns]
if missing: raise ValueError(f"Faltan columnas: {missing}")

data = raw[req].copy()
data["Date"] = pd.to_datetime(data["Date"], errors="coerce")
data["Voltage (V)"] = pd.to_numeric(data["Voltage (V)"], errors="coerce")
for c in reading_cols:
    data[c] = pd.to_numeric(data[c], errors="coerce")
data = data.dropna(subset=["Date", "Voltage (V)", *reading_cols]).copy()

# Fix 199 V fantasma
for d in sorted(data["Date"].unique()):
    sub = data.loc[data["Date"] == d]
    for v in sub.loc[sub.duplicated("Voltage (V)", keep=False), "Voltage (V)"].unique():
        idx = sub.index[sub["Voltage (V)"] == v].tolist()
        for k, i in enumerate(idx[1:], start=1):
            nv = v + k
            while ((data["Date"] == d) & (data["Voltage (V)"] == nv)).any():
                nv += 1
            data.loc[i, "Voltage (V)"] = float(nv)

data["n_repetitions"]     = data[reading_cols].notna().sum(axis=1)
data["P_mean_W"]          = data[reading_cols].mean(axis=1)
data["s_repeatability_W"] = data[reading_cols].std(axis=1, ddof=1)
data["u_repeatability_W"] = data["s_repeatability_W"] / np.sqrt(data["n_repetitions"])
data["u_calibration_W"]   = CERT_U_STANDARD_REL * data["P_mean_W"]
data["u_combined_W"]      = np.sqrt(data["u_repeatability_W"]**2 + data["u_calibration_W"]**2)
data["U_power_W"]         = RESULT_K * data["u_combined_W"]
data["U_power_relative_pct"] = 100 * data["U_power_W"] / data["P_mean_W"]
data = data.sort_values(["Date", "Voltage (V)"]).reset_index(drop=True)

assert not data.duplicated(["Date","Voltage (V)"]).any()
assert data["n_repetitions"].eq(5).all()

inv = data.groupby("Date")["Voltage (V)"].agg(
    n_voltajes="nunique", V_min="min", V_max="max").reset_index()
inv["Date"] = inv["Date"].dt.strftime("%d/%m/%Y")
show_colab_table(inv, "Inventario de sesiones")
register_table("Inventario de sesiones", inv)

In [ ]:
# --- Gráfico 01 ---
fig, ax = plt.subplots(figsize=(11, 7))
all_dates = sorted(data["Date"].unique())
cmap = plt.get_cmap("viridis")
for i, d in enumerate(all_dates):
    g = data.loc[data["Date"].eq(d)].sort_values("Voltage (V)")
    lab = pd.Timestamp(d).strftime("%d/%m/%Y") + (" (inicial)" if pd.Timestamp(d)==INITIAL_DATE else "")
    ax.errorbar(g["Voltage (V)"], g["P_mean_W"], yerr=g["U_power_W"],
                marker="o", ms=3.2, lw=1.2, capsize=2,
                color=cmap(i/max(1,len(all_dates)-1)), label=lab)
ax.set(title="Evolución potencia–tensión (todos los datos)",
       xlabel="Tensión (V)", ylabel="Potencia promedio (W)")
ax.legend(ncol=2, fontsize=8); fig.tight_layout()
p = FIG_DIR/"01_respuesta_global.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Evolución temporal potencia–tensión", p,
    "Curvas de potencia promedio vs tensión registrada para las 8 sesiones.")

## 3. Análisis de 9 puntos (23-may referencia)

In [ ]:
nine_voltages = np.sort(data.loc[data["Date"].eq(INITIAL_DATE),"Voltage (V)"].unique())
if len(nine_voltages) != 9: raise ValueError(f"Esperaba 9 voltajes, hay {len(nine_voltages)}")
nine_data = data.loc[data["Voltage (V)"].isin(nine_voltages)].copy()
counts = nine_data.groupby("Date")["Voltage (V)"].nunique()
bad = [pd.Timestamp(d).strftime("%d/%m/%Y") for d in data["Date"].unique() if counts.get(d,0)!=9]
if bad: raise ValueError(f"Fechas incompletas: {bad}")

base = (nine_data.loc[nine_data["Date"].eq(INITIAL_DATE),
        ["Voltage (V)","P_mean_W","u_repeatability_W","U_power_W"]]
        .rename(columns={"P_mean_W":"P0","u_repeatability_W":"u0","U_power_W":"U0"}))
nine_loss = nine_data.merge(base, on="Voltage (V)", how="left", validate="many_to_one")
nine_loss["ratio"]     = nine_loss["P_mean_W"] / nine_loss["P0"]
nine_loss["loss_pct"]  = 100 * (1 - nine_loss["ratio"])
nine_loss["u_ratio"]   = nine_loss["ratio"] * np.sqrt(
    (nine_loss["u_repeatability_W"]/nine_loss["P_mean_W"])**2 +
    (nine_loss["u0"]/nine_loss["P0"])**2)
nine_loss["U_loss_pct"] = RESULT_K * 100 * nine_loss["u_ratio"]
nine_loss.loc[nine_loss["Date"].eq(INITIAL_DATE), ["u_ratio","U_loss_pct"]] = 0.0

def summarize_loss(frame):
    rows = []
    for d, g in frame.groupby("Date", sort=True):
        u_std = np.sqrt(np.sum((g["U_loss_pct"]/RESULT_K)**2))/len(g)
        rows.append({"Date": pd.Timestamp(d), "n_volt": len(g),
                     "R_index": g["ratio"].mean(),
                     "U_R_index": RESULT_K*u_std/100,
                     "mean_loss_pct": g["loss_pct"].mean(),
                     "U_mean_loss_pct": RESULT_K*u_std})
    return pd.DataFrame(rows)

nine_summary = summarize_loss(nine_loss)
n9_disp = pd.DataFrame({
    "Fecha": nine_summary["Date"].dt.strftime("%d/%m/%Y"),
    "R9 ± U": [f"{v:.4f} ± {u:.4f}" for v,u in zip(nine_summary["R_index"], nine_summary["U_R_index"])],
    "Retenido (%)": (100*nine_summary["R_index"]).map(lambda v:f"{v:.2f}"),
    "Pérdida ± U (%)": [f"{l:.2f} ± {u:.2f}" for l,u in zip(nine_summary["mean_loss_pct"], nine_summary["U_mean_loss_pct"])],
})
show_colab_table(n9_disp, "Índice normalizado R9 (23-may = 1)")
register_table("Índice normalizado R9 (23-may = 1)", n9_disp)

In [ ]:
# --- Gráficos 02 y 03 ---
fig, ax = plt.subplots(figsize=(11, 7))
cmap = plt.get_cmap("viridis")
dates_9 = sorted(nine_data["Date"].unique())
for i, d in enumerate(dates_9):
    g = nine_data.loc[nine_data["Date"].eq(d)].sort_values("Voltage (V)")
    ax.errorbar(g["Voltage (V)"], g["P_mean_W"], yerr=g["U_power_W"],
                marker="o", lw=1.3, ms=4, capsize=2,
                color=cmap(i / max(1, len(dates_9) - 1)),
                label=pd.Timestamp(d).strftime("%d/%m/%Y"))
ax.set(title="Respuesta potencia–tensión en los 9 puntos iniciales",
       xlabel="Tensión registrada (V)", ylabel="Potencia promedio (W)")
ax.legend(ncol=2, fontsize=8); fig.tight_layout()
p = FIG_DIR/"02_respuesta_9puntos.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Respuesta potencia–tensión en 9 puntos", p,
    "Comparación entre sesiones en los 9 voltajes comunes al 23-may.")

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.errorbar(nine_summary["Date"], nine_summary["R_index"],
            yerr=nine_summary["U_R_index"],
            marker="o", capsize=3, lw=1.5, color="#176B87")
ax.axhline(1.0, color="#555555", ls="--", lw=1)
ax.set(title="Índice normalizado R9 (23-may = 1)",
       xlabel="Fecha", ylabel="R9")
ax.tick_params(axis="x", rotation=45); fig.tight_layout()
p = FIG_DIR/"03_indice_R9.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Índice normalizado R9 vs tiempo", p,
    "Evolución de la respuesta retenida, promedio de los 9 voltajes iniciales.")

## 4. Área bajo la curva 160–190 V

In [ ]:
complete_data = data.loc[data["Date"].ge(COMPLETE_BASELINE_DATE)].copy()
complete_dates = sorted(complete_data["Date"].unique())
vsets = [set(complete_data.loc[complete_data["Date"].eq(d),"Voltage (V)"]) for d in complete_dates]
common_v = np.array(sorted(set.intersection(*vsets)), dtype=float)
print(f"Sesiones: {len(complete_dates)} | Voltajes comunes: {len(common_v)} | "
      f"Rango: {common_v.min():.0f}–{common_v.max():.0f} V")

def trap_w(x):
    x = np.asarray(x, float)
    if len(x)<2 or np.any(np.diff(x)<=0): raise ValueError("Malla no creciente")
    dx = np.diff(x); w = np.zeros_like(x)
    w[0]=dx[0]/2; w[-1]=dx[-1]/2
    if len(x)>2: w[1:-1]=(dx[:-1]+dx[1:])/2
    return w

area_rows = []
area_src = complete_data.loc[complete_data["Voltage (V)"].between(AREA_V_MIN, AREA_V_MAX)].copy()
for d, g in area_src.groupby("Date", sort=True):
    g = g.sort_values("Voltage (V)")
    x = g["Voltage (V)"].to_numpy(float)
    y = g["P_mean_W"].to_numpy(float)
    ur= g["u_repeatability_W"].to_numpy(float)
    if not (np.isclose(x.min(), AREA_V_MIN) and np.isclose(x.max(), AREA_V_MAX)):
        raise ValueError(f"{pd.Timestamp(d):%d/%m/%Y} no cubre 160–190 V")
    w = trap_w(x)
    area = float(np.sum(w*y))
    u_rep = float(np.sqrt(np.sum((w*ur)**2)))
    u_cal = float(CERT_U_STANDARD_REL*area)
    u_tot = float(np.sqrt(u_rep**2 + u_cal**2))
    area_rows.append({"Date": pd.Timestamp(d), "n_pts": len(x),
                      "Area_WV": area, "u_area_rep": u_rep,
                      "u_area_cal": u_cal, "u_area_tot": u_tot,
                      "U_area": RESULT_K*u_tot})

area_summary = pd.DataFrame(area_rows)
base_row = area_summary.loc[area_summary["Date"].eq(COMPLETE_BASELINE_DATE)].iloc[0]
base_area, base_u = base_row["Area_WV"], base_row["u_area_rep"]

area_summary["A_norm"] = area_summary["Area_WV"] / base_area
area_summary["loss_global_pct"] = 100*(1 - area_summary["A_norm"])
area_summary["u_A_norm"] = area_summary["A_norm"]*np.sqrt(
    (area_summary["u_area_rep"]/area_summary["Area_WV"])**2 + (base_u/base_area)**2)
area_summary["U_A_norm"] = RESULT_K*area_summary["u_A_norm"]
area_summary["U_loss_global_pct"] = 100*area_summary["U_A_norm"]
area_summary.loc[area_summary["Date"].eq(COMPLETE_BASELINE_DATE),
                 ["u_A_norm","U_A_norm","U_loss_global_pct"]] = 0.0

area_disp = pd.DataFrame({
    "Fecha": area_summary["Date"].dt.strftime("%d/%m/%Y"),
    "Días desde inst.": (area_summary["Date"]-INSTALLATION_DATE).dt.days,
    "Área (W·V)": area_summary["Area_WV"].map(lambda v:f"{v:,.0f}"),
    "A_norm": area_summary["A_norm"].map(lambda v:f"{v:.4f}"),
    "Pérdida ± U (%)": [f"{l:.2f} ± {u:.2f}" for l,u in zip(
        area_summary["loss_global_pct"], area_summary["U_loss_global_pct"])],
})
show_colab_table(area_disp, "Área normalizada y pérdida global 160–190 V")
register_table("Área normalizada y pérdida global 160–190 V", area_disp)

In [ ]:
# --- Gráfico 07 ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
axes[0].errorbar(area_summary["Date"], area_summary["A_norm"],
                 yerr=area_summary["U_A_norm"], marker="o",
                 capsize=3, color="#176B87")
axes[0].axhline(1, color="#555555", ls="--", lw=1)
axes[0].set(title="Área normalizada 160–190 V", xlabel="Fecha",
            ylabel="Área normalizada")
axes[0].tick_params(axis="x", rotation=45)
axes[1].errorbar(area_summary["Date"], area_summary["loss_global_pct"],
                 yerr=area_summary["U_loss_global_pct"], marker="o",
                 capsize=3, color="#C8553D")
axes[1].axhline(0, color="#555555", ls="--", lw=1)
axes[1].set(title="Pérdida global de respuesta", xlabel="Fecha",
            ylabel="Pérdida global (%)")
axes[1].tick_params(axis="x", rotation=45)
fig.tight_layout()
p = FIG_DIR/"07_area_perdida.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Área normalizada y pérdida global", p,
    "Área normalizada (izq.) y pérdida relativa al 30-may (der.) con U expandida.")

## 5. Comparación de modelos — ÁREA

In [ ]:
area_md = area_summary.copy()
area_md["x_days"] = (area_md["Date"]-COMPLETE_BASELINE_DATE).dt.days.astype(float)
x_area = area_md["x_days"].to_numpy(float)
y_area = area_md["A_norm"].to_numpy(float)

def p_lin(p,x):    return 1 + p[0]*np.asarray(x,float)
def p_exp(p,x):    return np.exp(np.clip(p[0]*np.asarray(x,float),-50,50))
def p_quad(p,x):
    x = np.asarray(x,float); return 1 + p[0]*x + p[1]*x**2
def p_pow(p,x):
    x = np.maximum(np.asarray(x,float),0); return 1 - p[0]*x**p[1]
def p_log(p,x):
    A,k,x0 = p; x = np.asarray(x,float)
    return A + (1-A)*(1+np.exp(np.clip(-k*x0,-50,50)))/(1+np.exp(np.clip(k*(x-x0),-50,50)))
def p_gomp(p,x):
    A,b,c = p; x = np.asarray(x,float)
    return A + (1-A)*np.exp(np.clip(-b*(np.exp(np.clip(c*x,-50,50))-1),-50,50))
def p_weib(p,x):
    A,e,s = p; x = np.maximum(np.asarray(x,float),0)
    return A + (1-A)*np.exp(-np.clip((x/e)**s,0,700))
def p_semi(p,x):
    A,s = p; x = np.maximum(np.asarray(x,float),0)
    return A + (1-A)*np.exp(-0.5*(x/s)**2)

A_st = [0.0, 0.25, 0.45, 0.65]
cands = [
    {"Model":"Lineal",          "f":p_lin,  "starts":[[-0.005],[-0.01],[-0.02]],           "b":([-0.2],[0.05])},
    {"Model":"Exponencial",     "f":p_exp,  "starts":[[-0.005],[-0.01],[-0.03]],           "b":([-1.0],[0.05])},
    {"Model":"Cuadrático",      "f":p_quad, "starts":[[-0.005,-1e-4],[0,-3e-4],[-0.02,1e-4]],"b":([-0.2,-0.02],[0.1,0.02])},
    {"Model":"Ley de potencia", "f":p_pow,  "starts":[[0.005,1.0],[0.001,1.5],[0.02,0.7]],  "b":([1e-10,0.1],[10.0,4.0])},
    {"Model":"Logístico",       "f":p_log,  "starts":[[a,k,x0] for a in A_st for k in [0.03,0.1,0.3] for x0 in [15,28,40]],
     "b":([0.0,1e-5,-100.0],[0.99,2.0,200.0])},
    {"Model":"Gompertz",        "f":p_gomp, "starts":[[a,bb,c] for a in A_st for bb in [0.001,0.01,0.1] for c in [0.02,0.08,0.2]],
     "b":([0.0,1e-9,1e-9],[0.99,100.0,2.0])},
    {"Model":"Weibull",         "f":p_weib, "starts":[[a,e,s] for a in A_st for e in [15,30,50,100] for s in [0.5,1,2,5]],
     "b":([0.0,0.1,0.05],[0.99,1000.0,10.0])},
    {"Model":"Seminormal",      "f":p_semi, "starts":[[a,s] for a in A_st for s in [10,25,50,100]],
     "b":([0.0,0.1],[0.99,1000.0])},
]

def fit_model(spec, x, y):
    best = None
    for s in spec["starts"]:
        r = least_squares(lambda p: spec["f"](p,x)-y, s, bounds=spec["b"], max_nfev=100000)
        rss = float(np.sum(r.fun**2))
        if best is None or rss < best[0]: best = (rss, r)
    return best[1]

def aicc(rss, n, q):
    if n-q-1 <= 0: return np.inf
    return n*np.log(rss/n) + 2*q + 2*q*(q+1)/(n-q-1)

SST = float(np.sum((y_area-np.mean(y_area))**2))
rows, resid_rows, fits = [], [], {}
for spec in cands:
    r = fit_model(spec, x_area, y_area)
    fit = spec["f"](r.x, x_area)
    res = y_area - fit
    rss = float(np.sum(res**2))
    rmse = float(np.sqrt(rss/len(y_area)))
    r2 = 1 - rss/SST
    q = len(r.x)
    r2a = 1 - (1-r2)*(len(y_area)-1)/(len(y_area)-q)
    rows.append({"Model":spec["Model"], "q":q, "RMSE":rmse, "R2":r2,
                 "R2_adj":r2a, "RSS":rss})
    fits[spec["Model"]] = {"spec":spec, "res":r}
    for d,o,p,rr in zip(area_md["Date"], y_area, fit, res):
        resid_rows.append({"Model":spec["Model"], "Date":d,
                           "Observed":o, "Predicted":p, "Residual":rr})

area_cmp = pd.DataFrame(rows)
area_cmp["AICc"] = [aicc(r, len(y_area), q) for r,q in zip(area_cmp["RSS"], area_cmp["q"])]
area_cmp["dAICc"] = area_cmp["AICc"] - area_cmp["AICc"].min()
area_cmp = area_cmp.sort_values("AICc").reset_index(drop=True)
area_res = pd.DataFrame(resid_rows)

def loo_pred(spec, x, y):
    n = len(x); out = np.full(n, np.nan)
    for i in range(n):
        m = np.arange(n)!=i
        try:
            r = fit_model(spec, x[m], y[m])
            out[i] = float(spec["f"](r.x, np.array([x[i]]))[0])
        except Exception: pass
    return out

loo_rows = []
for spec in cands:
    pr = loo_pred(spec, x_area, y_area)
    v = ~np.isnan(pr)
    if v.sum()<3: continue
    res = y_area[v]-pr[v]
    loo_rows.append({"Model":spec["Model"],
                     "LOO_RMSE":float(np.sqrt(np.mean(res**2))),
                     "LOO_MAE":float(np.mean(np.abs(res)))})
loo_df = pd.DataFrame(loo_rows).sort_values("LOO_RMSE").reset_index(drop=True)
area_cmp = area_cmp.merge(loo_df, on="Model", how="left")

area_disp = pd.DataFrame({
    "Modelo": area_cmp["Model"],
    "q": area_cmp["q"].astype(int),
    "RMSE": area_cmp["RMSE"].map(lambda v:f"{v:.5f}"),
    "R²": area_cmp["R2"].map(lambda v:f"{v:.4f}"),
    "R² ajustado": area_cmp["R2_adj"].map(lambda v:f"{v:.4f}"),
    "AICc": area_cmp["AICc"].map(lambda v:f"{v:.3f}"),
    "ΔAICc": area_cmp["dAICc"].map(lambda v:f"{v:.3f}"),
    "LOO RMSE": area_cmp["LOO_RMSE"].map(lambda v:"—" if pd.isna(v) else f"{v:.5f}"),
})
show_colab_table(area_disp, "R², R²aj, RMSE, AICc y LOO — modelos del ÁREA")
register_table("Comparación de modelos del ÁREA normalizada", area_disp,
    "Todos los modelos anclados a A_norm(0)=1. Menor AICc/RMSE/LOO indica mejor ajuste.")

winner_area_aicc = area_cmp.iloc[0]["Model"]
winner_area_loo  = area_cmp.sort_values("LOO_RMSE").iloc[0]["Model"]
print(f"Ganador por AICc: {winner_area_aicc}")
print(f"Ganador por LOO:  {winner_area_loo}")

In [ ]:
# --- Gráfico 08 ---
x_dense = np.linspace(x_area.min(), x_area.max(), 500)
x_area_disp  = x_area  + DAYS_OFFSET
x_dense_disp = x_dense + DAYS_OFFSET

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
cmap = plt.get_cmap("tab10")

axes[0].errorbar(x_area_disp, y_area, yerr=area_md["U_A_norm"], fmt="o",
                 color="black", ecolor="#555", capsize=3,
                 label="Observado ± U", zorder=5)
for i, spec in enumerate(cands):
    axes[0].plot(x_dense_disp,
                 spec["f"](fits[spec["Model"]]["res"].x, x_dense),
                 lw=1.5, color=cmap(i), label=spec["Model"])
axes[0].set(xlabel="Días desde instalación (19-may-2026)",
            ylabel="Área normalizada 160–190 V",
            title="Ajustes"); axes[0].legend(fontsize=7, ncol=2)

for i, m in enumerate(area_cmp["Model"]):
    s = area_res.loc[area_res["Model"].eq(m)]
    axes[1].plot(x_area_disp, s["Residual"], "o-", ms=3.5, lw=1.2,
                 color=cmap(i), label=m)
axes[1].axhline(0, color="k", ls="--", lw=1)
axes[1].set(xlabel="Días desde instalación (19-may-2026)", ylabel="Residuo",
            title="Residuos"); axes[1].legend(fontsize=7, ncol=2)

oord_ = area_cmp.sort_values("RMSE").reset_index(drop=True)
axes[2].barh(ord_["Model"], ord_["RMSE"], color="#176B87")
axes[2].invert_yaxis()
axes[2].set(xlabel="RMSE", title="Error en muestra")
axes[2].set_yticks(range(len(ord_)))
axes[2].set_yticklabels(ord_["Model"])
fig.suptitle("Modelos de degradación del área normalizada", fontsize=14)
fig.tight_layout()
p = FIG_DIR/"08_modelos_area.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Modelos de degradación del ÁREA normalizada", p,
    "Ajustes (izq.), residuos (centro) y RMSE ordenado (der.) para los 8 modelos. "
    "Eje temporal en días desde la instalación (19-may-2026).")

## 6. V6 — tensión requerida para 6 W

In [ ]:
def interp_v6(session, target=TARGET_POWER_W):
    s = session.sort_values("Voltage (V)").reset_index(drop=True)
    V = s["Voltage (V)"].to_numpy(float); P = s["P_mean_W"].to_numpy(float)
    ur= s["u_repeatability_W"].to_numpy(float)
    br = [i for i in range(len(s)-1) if P[i] <= target <= P[i+1] and P[i+1]>P[i]]
    if not br: return None
    i = br[0]
    v1,v2,p1,p2 = V[i],V[i+1],P[i],P[i+1]
    dP, dV = p2-p1, v2-v1
    v6 = v1 + (target-p1)*dV/dP
    d1 = dV*(target-p2)/dP**2
    d2 = -dV*(target-p1)/dP**2
    u_rep = np.sqrt((d1*ur[i])**2 + (d2*ur[i+1])**2)
    u_cal = abs(CERT_U_STANDARD_REL*(d1*p1 + d2*p2))
    u_tot = np.sqrt(u_rep**2 + u_cal**2)
    return {"V1_V":v1,"P1_W":p1,"V2_V":v2,"P2_W":p2,"V6_V":v6,
            "u_V6_rep":u_rep,"u_V6_cal":u_cal,"u_V6_tot":u_tot,"U_V6":RESULT_K*u_tot}

v6_rows, no_bracket = [], []
for d, s in data.groupby("Date", sort=True):
    r = interp_v6(s)
    if r is None: no_bracket.append(pd.Timestamp(d)); continue
    r["Date"] = pd.Timestamp(d); v6_rows.append(r)

v6_summary = pd.DataFrame(v6_rows).sort_values("Date").reset_index(drop=True)
v6_summary["x_days"] = (v6_summary["Date"]-COMPLETE_BASELINE_DATE).dt.days.astype(float)
v6_summary["days_inst"] = (v6_summary["Date"]-INSTALLATION_DATE).dt.days.astype(float)
v6_summary["V6_lower"] = v6_summary["V6_V"] - v6_summary["U_V6"]
v6_summary["Confirmed_below6W"] = v6_summary["V6_lower"] > OPERATING_V_MAX

v6_disp = pd.DataFrame({
    "Fecha": v6_summary["Date"].dt.strftime("%d/%m/%Y"),
    "Días inst.": v6_summary["days_inst"].astype(int),
    "V6 ± U (V)": [f"{v:.3f} ± {u:.3f}" for v,u in zip(v6_summary["V6_V"], v6_summary["U_V6"])],
    "¿<6W confirmado?": np.where(v6_summary["Confirmed_below6W"], "Sí", "No"),
})
show_colab_table(v6_disp, "V6 por sesión")
register_table("V6: tensión requerida para alcanzar 6 W", v6_disp)

In [ ]:
x_v6 = v6_summary["x_days"].to_numpy(float)
y_v6 = v6_summary["V6_V"].to_numpy(float)
v6_0 = float(y_v6[0])

def v6_lin(p,x):   return v6_0 + p[0]*np.asarray(x,float)
def v6_exp(p,x):   return v6_0*np.exp(np.clip(p[0]*np.asarray(x,float),-50,50))
def v6_quad(p,x):
    x = np.asarray(x,float); return v6_0 + p[0]*x + p[1]*x**2
def v6_pow(p,x):
    x = np.maximum(np.asarray(x,float),0); return v6_0 + p[0]*x**p[1]
def v6_log(p,x):
    A,k,x0 = p; x = np.asarray(x,float)
    return A - (A-v6_0)*(1+np.exp(np.clip(-k*x0,-50,50)))/(1+np.exp(np.clip(k*(x-x0),-50,50)))
def v6_gomp(p,x):
    A,b,c = p; x = np.asarray(x,float)
    return A - (A-v6_0)*np.exp(np.clip(-b*(np.exp(np.clip(c*x,-50,50))-1),-50,50))
def v6_weib(p,x):
    A,e,s = p; x = np.maximum(np.asarray(x,float),0)
    return A - (A-v6_0)*np.exp(-np.clip((x/e)**s,0,700))
def v6_semi(p,x):
    A,s = p; x = np.maximum(np.asarray(x,float),0)
    return A - (A-v6_0)*np.exp(-0.5*(x/s)**2)

asym = [185, 195, 220, 260]
v6_cands = [
    {"Model":"Lineal",          "f":v6_lin,  "starts":[[0.1],[0.3],[0.6]],           "b":([-5.0],[5.0])},
    {"Model":"Exponencial",     "f":v6_exp,  "starts":[[0.001],[0.003],[0.008]],     "b":([-0.1],[0.1])},
    {"Model":"Cuadrático",      "f":v6_quad, "starts":[[0,0.005],[0.1,0.01],[0.3,0.005],[-0.1,0.02]],"b":([-5,-0.2],[5,0.2])},
    {"Model":"Ley de potencia", "f":v6_pow,  "starts":[[0.05,1],[0.01,1.5],[0.001,2],[0.5,0.7]],"b":([1e-9,0.1],[50.0,4.0])},
    {"Model":"Logístico",       "f":v6_log,  "starts":[[a,k,x0] for a in asym for k in [0.03,0.1,0.3] for x0 in [15,28,40]],
     "b":([v6_0+0.01,1e-5,-200.0],[400.0,2.0,300.0])},
    {"Model":"Gompertz",        "f":v6_gomp, "starts":[[a,bb,c] for a in asym for bb in [0.001,0.01,0.1] for c in [0.02,0.08,0.2]],
     "b":([v6_0+0.01,1e-9,1e-9],[400.0,100.0,2.0])},
    {"Model":"Weibull",         "f":v6_weib, "starts":[[a,e,s] for a in asym for e in [15,30,50,100] for s in [0.5,1,2,5]],
     "b":([v6_0+0.01,0.1,0.05],[400.0,1000.0,10.0])},
    {"Model":"Seminormal",      "f":v6_semi, "starts":[[a,s] for a in asym for s in [10,25,50,100]],
     "b":([v6_0+0.01,0.1],[400.0,1000.0])},
]

def fit_v6(spec, x, y):
    best = None
    for s in spec["starts"]:
        r = least_squares(lambda p: spec["f"](p,x)-y, s, bounds=spec["b"], max_nfev=100000)
        rss = float(np.sum(r.fun**2))
        if best is None or rss < best[0]: best = (rss, r)
    return best[1]

def first_cross(spec, p, level=OPERATING_V_MAX, horizon=500.0):
    grid = np.linspace(0, horizon, 10001)
    diff = spec["f"](p, grid) - level
    for i in range(len(grid)-1):
        if diff[i] == 0: return float(grid[i])
        if diff[i] < 0 <= diff[i+1]:
            return float(brentq(lambda v: float(spec["f"](p, np.array([v]))[0]-level),
                                grid[i], grid[i+1]))
    return np.nan

SST_v6 = float(np.sum((y_v6-np.mean(y_v6))**2))
v6_rows_cmp, v6_fits = [], {}
for spec in v6_cands:
    r = fit_v6(spec, x_v6, y_v6)
    fit = spec["f"](r.x, x_v6)
    res = y_v6 - fit
    rss = float(np.sum(res**2))
    q = len(r.x)
    rmse = float(np.sqrt(rss/len(y_v6)))
    r2 = 1 - rss/SST_v6
    r2a = 1 - (1-r2)*(len(y_v6)-1)/(len(y_v6)-q)
    xc = first_cross(spec, r.x)
    v6_rows_cmp.append({"Model":spec["Model"],"q":q,"RMSE":rmse,
                        "R2":r2,"R2_adj":r2a,"RSS":rss,"cross_day":xc})
    v6_fits[spec["Model"]] = {"spec":spec, "res":r}

v6_cmp = pd.DataFrame(v6_rows_cmp)
v6_cmp["AICc"] = [aicc(r, len(y_v6), q) for r,q in zip(v6_cmp["RSS"], v6_cmp["q"])]
v6_cmp["dAICc"] = v6_cmp["AICc"] - v6_cmp["AICc"].min()
v6_cmp = v6_cmp.sort_values("AICc").reset_index(drop=True)

loo_v6_rows = []
for spec in v6_cands:
    pr = np.full(len(x_v6), np.nan)
    for i in range(len(x_v6)):
        m = np.arange(len(x_v6)) != i
        try:
            r = fit_v6(spec, x_v6[m], y_v6[m])
            pr[i] = float(spec["f"](r.x, np.array([x_v6[i]]))[0])
        except Exception: pass
    v = ~np.isnan(pr)
    if v.sum() < 3: continue
    res = y_v6[v] - pr[v]
    loo_v6_rows.append({"Model": spec["Model"],
                        "LOO_RMSE": float(np.sqrt(np.mean(res**2))),
                        "LOO_MAE": float(np.mean(np.abs(res)))})
loo_v6_df = pd.DataFrame(loo_v6_rows).sort_values("LOO_RMSE").reset_index(drop=True)
v6_cmp = v6_cmp.merge(loo_v6_df, on="Model", how="left")

v6_disp = pd.DataFrame({
    "Modelo": v6_cmp["Model"],
    "q": v6_cmp["q"].astype(int),
    "RMSE (V)": v6_cmp["RMSE"].map(lambda v:f"{v:.3f}"),
    "R²": v6_cmp["R2"].map(lambda v:f"{v:.4f}"),
    "R² ajustado": v6_cmp["R2_adj"].map(lambda v:f"{v:.4f}"),
    "AICc": v6_cmp["AICc"].map(lambda v:f"{v:.3f}"),
    "ΔAICc": v6_cmp["dAICc"].map(lambda v:f"{v:.3f}"),
    "LOO RMSE": v6_cmp["LOO_RMSE"].map(lambda v:"—" if pd.isna(v) else f"{v:.3f}"),
    "Cruce 190 V": v6_cmp["cross_day"].map(
    lambda v: "No cruza" if pd.isna(v) else f"{v + DAYS_OFFSET:.1f} d"),
})
show_colab_table(v6_disp, "R², R²aj, RMSE, AICc y LOO — modelos de V6")
register_table("Comparación de modelos de V6", v6_disp,
    "V6 = tensión interpolada para 6 W. Cruce 190 V indica el día (desde la instalación, 19-may-2026) "
    "en que el modelo predice que 190 V dejan de ser suficientes.")

In [ ]:
# =============================================================
# SELECCIÓN DEL MODELO PARA V6 — Seminormal (justificado por LOO)
# =============================================================
# Razonamiento:
# - Los modelos de 3 parámetros (Gompertz/Weibull/Logístico) ajustan su
#   asíntota exactamente al último dato observado (~183.9 V), lo cual es
#   sospechoso y contradice la evidencia de crecimiento continuo.
# - Seminormal obtiene el MEJOR LOO RMSE (capacidad predictiva empírica)
#   entre todos los modelos evaluados.
# - Su asíntota (219 V) es más coherente con la física del envejecimiento
#   y permite que V6 siga creciendo si la degradación persiste.

sel_name = V6_MODEL_OVERRIDE
print(f"Modelo seleccionado para V6: {sel_name}")
print(f"Justificación: menor LOO RMSE = {v6_cmp.loc[v6_cmp['Model'].eq(sel_name), 'LOO_RMSE'].iloc[0]:.3f} V")

# Comparar AICc y LOO para todos los modelos que cruzan
print("\n=== Modelos que cruzan 190 V ===")
crossers = v6_cmp.dropna(subset=["cross_day"]).sort_values("LOO_RMSE")
for _, r in crossers.iterrows():
    print(f"  {r['Model']:<15s}  ΔAICc={r['dAICc']:>6.2f}  LOO={r['LOO_RMSE']:.3f}  "
      f"día={r['cross_day'] + DAYS_OFFSET:.1f}")

In [ ]:
# --- Gráfico 09 ---
HORIZON_PLOT = 4 * float(x_v6.max() - x_v6.min())
x_plot = np.linspace(0, HORIZON_PLOT, 500)
x_plot_disp = x_plot + DAYS_OFFSET
x_v6_disp   = x_v6   + DAYS_OFFSET

fig, axes = plt.subplots(1, 2, figsize=(15.5, 6))

# ---------- Panel izquierdo ----------
axes[0].errorbar(x_v6_disp, y_v6, yerr=v6_summary["U_V6"],
                 fmt="o", color="black", ecolor="#555555",
                 capsize=3, label="V6 observado ± U", zorder=5)
cmap = plt.get_cmap("tab10")
for i, spec in enumerate(v6_cands):
    axes[0].plot(x_plot_disp,
                 spec["f"](v6_fits[spec["Model"]]["res"].x, x_plot),
                 lw=1.5, color=cmap(i), label=spec["Model"])
axes[0].axhline(OPERATING_V_MAX, color="#C23B22", ls="--", lw=1.5,
                label="Límite 190 V")
axes[0].axvline(x_v6.max() + DAYS_OFFSET, color="#777777", ls=":", lw=1.2,
                label="Última medición")
axes[0].set(title="Comparación de 8 modelos no segmentados",
            xlabel="Días desde la instalación (19-may-2026)",
            ylabel="V6 (V)")
axes[0].set_ylim(165, 215); axes[0].legend(fontsize=7, ncol=2)

# ---------- Panel derecho ----------
sel_r = v6_fits[sel_name]["res"]
spec_sel = v6_fits[sel_name]["spec"]
df_res = len(y_v6) - len(sel_r.x)
resid_var = float(np.sum(sel_r.fun**2) / df_res)
cov = resid_var * np.linalg.pinv(sel_r.jac.T @ sel_r.jac)
t_crit = float(student_t.ppf(0.975, df=df_res))

def pred_band(spec, params, x_arr):
    mean = spec["f"](params, x_arr)
    grad = np.zeros((len(x_arr), len(params)))
    for k in range(len(params)):
        step = 1e-5 * max(1.0, abs(params[k]))
        up, lo = params.copy(), params.copy()
        up[k] += step; lo[k] -= step
        grad[:, k] = (spec["f"](up, x_arr) - spec["f"](lo, x_arr)) / (2*step)
    mvar = np.einsum("ij,jk,ik->i", grad, cov, grad)
    se_mean = np.sqrt(np.maximum(mvar, 0))
    se_pred = np.sqrt(resid_var + np.maximum(mvar, 0))
    return (mean, mean - t_crit*se_mean, mean + t_crit*se_mean,
            mean - t_crit*se_pred, mean + t_crit*se_pred)

mean, mlo, mhi, plo, phi = pred_band(spec_sel, sel_r.x, x_plot)

axes[1].fill_between(x_plot_disp, plo, phi, color="#4C78A8", alpha=0.15, label="IP 95 %")
axes[1].fill_between(x_plot_disp, mlo, mhi, color="#4C78A8", alpha=0.30, label="IC del modelo 95 %")
axes[1].plot(x_plot_disp, mean, color="#4C78A8", lw=2.2, label=f"{sel_name} ajustado")
axes[1].errorbar(x_v6_disp, y_v6, yerr=v6_summary["U_V6"],
                 fmt="o", color="black", ecolor="#555555",
                 capsize=3, label="V6 observado ± U", zorder=5)
axes[1].axhline(OPERATING_V_MAX, color="#C23B22", ls="--", lw=1.5, label="Límite 190 V")
axes[1].axvline(x_v6.max() + DAYS_OFFSET, color="#777777", ls=":", lw=1.2, label="Última medición")

xc = v6_cmp.loc[v6_cmp["Model"].eq(sel_name), "cross_day"].iloc[0]
if pd.notna(xc) and xc <= x_plot.max():
    axes[1].axvline(xc + DAYS_OFFSET, color="#C23B22", ls=":", lw=1.4,
                    label=f"Cruce nominal: d={xc + DAYS_OFFSET:.1f}")

axes[1].set(title=f"Extrapolación del modelo seleccionado ({sel_name})",
            xlabel="Días desde la instalación (19-may-2026)",
            ylabel="V6 (V)")
axes[1].set_ylim(165, 225); axes[1].legend(fontsize=7, loc="lower right")

fig.suptitle("Extrapolación del momento en que 190 V dejan de ser suficientes",
             fontsize=14)
fig.tight_layout()
p = FIG_DIR/"09_extrapolacion_v6.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Modelos y extrapolación de V6", p,
    f"Izquierda: 8 modelos ajustados a V6(t). Derecha: {sel_name} seleccionado "
    f"(menor LOO RMSE) con IC 95 %. Eje temporal en días desde la instalación (19-may-2026).")

## 7. Bootstrap del día de cruce

In [ ]:
OBS_RANGE = float(x_v6.max() - x_v6.min())
HORIZON   = 4 * OBS_RANGE
print(f"Rango observado: {OBS_RANGE:.0f} d | Horizonte: {HORIZON:.0f} d (4x)")

sel = v6_fits[sel_name]
sel_res = y_v6 - sel["spec"]["f"](sel["res"].x, x_v6)

def boot_crossing(spec, x, y, res, n_boot=2000, horizon=HORIZON, seed=42):
    rng = np.random.default_rng(seed)
    days = []
    for _ in range(n_boot):
        yb = y + rng.choice(res, size=len(y), replace=True)
        try:
            r = fit_v6(spec, x, yb)
            d = first_cross(spec, r.x, OPERATING_V_MAX, horizon)
            if np.isfinite(d): days.append(d)
        except Exception: pass
    return np.array(days)

boot_days = boot_crossing(sel["spec"], x_v6, y_v6, sel_res)

if len(boot_days) > 0:
    q05, q50, q95 = np.percentile(boot_days, [5, 50, 95])
    print(f"Bootstrap ({len(boot_days)} muestras):")
    print(f"  Mediana: día {q50 + DAYS_OFFSET:.1f} (desde instalación)")
    print(f"  IC 90%: [{q05 + DAYS_OFFSET:.1f}, {q95 + DAYS_OFFSET:.1f}]  (ancho {q95 - q05:.1f} d)")
    print(f"  Fecha mediana: "
          f"{(COMPLETE_BASELINE_DATE + pd.to_timedelta(q50, unit='D')):%d/%m/%Y}")
    print(f"  IC fechas: "
          f"{(COMPLETE_BASELINE_DATE + pd.to_timedelta(q05, unit='D')):%d/%m/%Y} – "
          f"{(COMPLETE_BASELINE_DATE + pd.to_timedelta(q95, unit='D')):%d/%m/%Y}")

    boot_summary = pd.DataFrame({
        "Estadístico": ["Mediana", "Percentil 5", "Percentil 95", "Ancho IC 90 %"],
        "Día desde instalación (19-may)": [
            f"{q50 + DAYS_OFFSET:.1f}",
            f"{q05 + DAYS_OFFSET:.1f}",
            f"{q95 + DAYS_OFFSET:.1f}",
            f"{q95 - q05:.1f}",
        ],
        "Fecha": [
            f"{(COMPLETE_BASELINE_DATE + pd.to_timedelta(q50, unit='D')):%d/%m/%Y}",
            f"{(COMPLETE_BASELINE_DATE + pd.to_timedelta(q05, unit='D')):%d/%m/%Y}",
            f"{(COMPLETE_BASELINE_DATE + pd.to_timedelta(q95, unit='D')):%d/%m/%Y}",
            "—",
        ],
    })
    show_colab_table(boot_summary, f"Bootstrap del día de cruce — {sel_name}")
    register_table(f"Bootstrap del día de cruce — modelo {sel_name}", boot_summary,
        f"Basado en {len(boot_days)} réplicas bootstrap del modelo {sel_name}. "
        f"Este intervalo cuantifica solo la incertidumbre sobre los parámetros "
        f"del modelo seleccionado, NO la incertidumbre sobre la selección del modelo.")

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.hist(boot_days + DAYS_OFFSET, bins=40, color="#4C78A8", alpha=0.75, edgecolor="white")
    ax.axvline(q50 + DAYS_OFFSET, color="k", lw=2,
               label=f"Mediana {q50 + DAYS_OFFSET:.1f} d")
    ax.axvline(q05 + DAYS_OFFSET, color="red", ls="--", lw=1.5,
               label=f"P5 {q05 + DAYS_OFFSET:.1f} d")
    ax.axvline(q95 + DAYS_OFFSET, color="red", ls="--", lw=1.5,
               label=f"P95 {q95 + DAYS_OFFSET:.1f} d")
    ax.axvline(x_v6.max() + DAYS_OFFSET, color="#777", ls=":", lw=1.2,
               label=f"Última medición ({x_v6.max() + DAYS_OFFSET:.0f} d)")
    ax.set(xlabel="Día de cruce 190 V (desde instalación)", ylabel="Frecuencia",
           title=f"Bootstrap del día de cruce — {sel_name}")
    ax.legend(); fig.tight_layout()
    p = FIG_DIR/"11_bootstrap_cruce.png"
    fig.savefig(p, dpi=220); plt.show()
    register_figure("Bootstrap del día de cruce", p,
        "Distribución del día (desde la instalación) en que V6 alcanza 190 V bajo remuestreo de residuos.")
else:
    print("Bootstrap no produjo cruces → la fecha nominal NO es robusta.")

# --- Advertencia metodológica ---
warning_text = f'''
METODOLOGÍA Y ADVERTENCIAS:

El modelo {sel_name} fue seleccionado por presentar el menor LOO RMSE
({v6_cmp.loc[v6_cmp["Model"].eq(sel_name), "LOO_RMSE"].iloc[0]:.3f} V) entre todos
los modelos evaluados. Los modelos de 3 parámetros (Gompertz, Weibull, Logístico)
obtuvieron mejor AICc pero ajustan su asíntota exactamente al último dato observado
(~183.9 V), lo cual es sospechoso y contradice la evidencia de crecimiento continuo.

IMPORTANTE: El intervalo bootstrap anterior cuantifica únicamente la incertidumbre
sobre los parámetros del modelo {sel_name}. NO incluye la incertidumbre sobre
si {sel_name} es efectivamente el modelo correcto. La incertidumbre real es mayor.

Se recomienda confirmar con 2-3 sesiones adicionales para verificar si V6 continúa
creciendo por encima de 184 V (falsa saturación) o se estabiliza (saturación real).
'''
print(warning_text)
register_text("Advertencia metodológica sobre la selección del modelo", warning_text)

## 8. Desecho semanal

In [ ]:
ws = pd.read_excel(data_file, sheet_name="Scrap Data by Week")
ws.columns = [str(c).strip() for c in ws.columns]
ws = ws.rename(columns={
    "Week":"Week", "Date Range":"Date_range",
    "Corte Irregular":"Irregular_cut_count",
    "Longitud fuera de especificación":"Length_out_count",
    "Dimensión fuera de especificación":"Dimension_out_count",
    "Profundidad de Scribe fuera de especificación":"Scribe_depth_out_count",
    "Total":"Nonconforming_count"})

dl = pd.read_excel(data_file, sheet_name="Production Data by Week")
dl.columns = [str(c).strip() for c in dl.columns]
dl = dl.rename(columns={"Weeks Number":"Week","Total Quatity":"Production_total"})
dl["Week"] = pd.to_numeric(dl["Week"], errors="coerce")
dl["Production_total"] = pd.to_numeric(dl["Production_total"], errors="coerce")
dl = dl.dropna(subset=["Week","Production_total"]).copy()
dl["Week"] = dl["Week"].astype(int)

count_cols = ["Irregular_cut_count","Length_out_count","Dimension_out_count",
              "Scribe_depth_out_count","Nonconforming_count"]
ws["Week"] = pd.to_numeric(ws["Week"], errors="coerce")
ws = ws.dropna(subset=["Week"]).copy(); ws["Week"] = ws["Week"].astype(int)
for c in count_cols: ws[c] = pd.to_numeric(ws[c], errors="coerce")

wq = (ws.merge(dl, on="Week", how="inner", validate="one_to_one")
        .loc[lambda d: d["Week"].between(21,28)]
        .sort_values("Week").reset_index(drop=True))
wq["q"] = wq["Nonconforming_count"]/wq["Production_total"]
wq["q_pct"] = 100*wq["q"]

z = 1.959963984540054
qq = wq["q"].to_numpy(float); nn = wq["Production_total"].to_numpy(float)
den = 1 + z**2/nn
c_ = (qq + z**2/(2*nn))/den
hw = z*np.sqrt(qq*(1-qq)/nn + z**2/(4*nn**2))/den
wq["W_lo"] = 100*(c_-hw); wq["W_hi"] = 100*(c_+hw)

opt = (nine_summary[["Date","R_index","U_R_index"]]
    .rename(columns={"R_index":"R9","U_R_index":"U_R9"})
    .merge(area_summary[["Date","A_norm","U_A_norm","loss_global_pct","U_loss_global_pct"]],
           on="Date", how="left")
    .merge(v6_summary[["Date","V6_V","U_V6"]], on="Date", how="left"))
opt["Week"] = opt["Date"].dt.isocalendar().week.astype(int)
wa = wq.merge(opt, on="Week", how="left", validate="one_to_one")
wa["R9_loss"] = 100*(1 - wa["R9"])

wa_disp = pd.DataFrame({
    "Semana": wa["Week"], "Rango": wa["Date_range"],
    "Fecha óptica": wa["Date"].dt.strftime("%d/%m/%Y"),
    "D": wa["Nonconforming_count"].astype(int),
    "N": wa["Production_total"].astype(int),
    "q (%) ± IC95": [f"{v:.3f} [{l:.3f}, {u:.3f}]" for v,l,u in zip(wa["q_pct"],wa["W_lo"],wa["W_hi"])],
    "Pérdida área (%)": ["—" if pd.isna(v) else f"{v:.2f}" for v in wa["loss_global_pct"]],
    "V6 ± U": ["—" if pd.isna(v) else f"{v:.2f} ± {u:.2f}" for v,u in zip(wa["V6_V"],wa["U_V6"])],
})
show_colab_table(wa_disp, "Desecho vs condición óptica")
register_table("Desecho semanal vs condición óptica", wa_disp)

## 9. Correlaciones con desecho

In [ ]:
inf = wa.loc[wa["Week"].between(22,28)].copy().reset_index(drop=True)
q_vec = inf["q"].to_numpy(float)
t_vec = inf["Week"].to_numpy(float)

def exact_spearman(x, y):
    x = np.asarray(x,float); y = np.asarray(y,float)
    obs = float(spearmanr(x,y).statistic)
    c = t = 0
    for p in itertools.permutations(y):
        r = float(spearmanr(x,p).statistic)
        c += abs(r) >= abs(obs)-1e-12; t += 1
    return obs, c/t

def partial_corr(x, y, z):
    x, y, z = map(lambda v: np.asarray(v,float), (x,y,z))
    def resid(a, b):
        B = np.column_stack([np.ones(len(b)), b])
        c, *_ = np.linalg.lstsq(B, a, rcond=None)
        return a - B@c
    rx, ry = resid(x,z), resid(y,z)
    r, p = pearsonr(rx, ry)
    return r, p

def boot_corr(x, y, method="spearman", n_boot=5000, seed=42):
    rng = np.random.default_rng(seed)
    x, y = np.asarray(x,float), np.asarray(y,float)
    n = len(x); out = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(x[idx]))<3 or len(np.unique(y[idx]))<3: continue
        if method=="spearman": r = spearmanr(x[idx],y[idx]).statistic
        elif method=="kendall": r = kendalltau(x[idx],y[idx]).statistic
        else: r = np.corrcoef(x[idx],y[idx])[0,1]
        if np.isfinite(r): out.append(r)
    return np.array(out)

indicators = [
    ("Pérdida de área 160–190 V", inf["loss_global_pct"].to_numpy(float)),
    ("V6", inf["V6_V"].to_numpy(float)),
    ("Pérdida R9", inf["R9_loss"].to_numpy(float)),
]

corr_rows = []
for label, vals in indicators:
    rho, p_rho = spearmanr(vals, q_vec)
    tau, p_tau = kendalltau(vals, q_vec)
    r_par, p_par = partial_corr(vals, q_vec, t_vec)
    b = boot_corr(vals, q_vec, "spearman")
    lo, med, hi = np.percentile(b, [2.5, 50, 97.5])
    try:
        r_pear, p_pear = pearsonr(vals, q_vec)
    except Exception:
        r_pear, p_pear = np.nan, np.nan
    d = dcor.distance_correlation(vals, q_vec) if DCOR_OK else np.nan
    corr_rows.append({
        "Indicador": label,
        "Spearman ρ (p)": f"{rho:.3f} ({p_rho:.3f})",
        "Kendall τ (p)":  f"{tau:.3f} ({p_tau:.3f})",
        "Parcial|tiempo (r, p)": f"{r_par:.3f} ({p_par:.3f})",
        "Pearson r (p)":  f"{r_pear:.3f} ({p_pear:.3f})",
        "IC 95% bootstrap": f"[{lo:.3f}, {hi:.3f}]",
        "¿IC cubre 0?": "Sí" if lo<=0<=hi else "No",
        "Distance corr": f"{d:.3f}" if np.isfinite(d) else "—",
    })

corr_df = pd.DataFrame(corr_rows)
show_colab_table(corr_df, "Comparación de métodos de correlación")
register_table("Correlación desecho vs indicadores ópticos", corr_df,
    "La correlación parcial controlada por tiempo revela que la asociación cruda es espuria.")

In [ ]:
# --- Gráfico 10 (versión simple, sin banda ni error bars en desecho) ---
fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
weeks = wa["Week"].to_numpy(int)
scrap = wa["q_pct"].to_numpy(float)

# --- Panel 1: evolución semanal simple ---
ax0 = axes[0]
ax0b = ax0.twinx()

# Desecho (sin error bars, sin banda)
ax0.plot(weeks, scrap, "o-", color="#C23B22", lw=1.8,
         markersize=6, label="Desecho")

# Pérdida de área (eje derecho)
cw = wa.dropna(subset=["loss_global_pct"])
ax0b.plot(cw["Week"], cw["loss_global_pct"], "s-", color="#176B87",
          lw=1.8, markersize=6, label="Pérdida área")

ax0.set(xlabel="Semana ISO", ylabel="Desecho (%)",
        title="Evolución semanal")
ax0b.set_ylabel("Pérdida área (%)", color="#176B87")
ax0.tick_params(axis="y", colors="#C23B22")
ax0b.tick_params(axis="y", colors="#176B87")

# Leyenda combinada
h1, l1 = ax0.get_legend_handles_labels()
h2, l2 = ax0b.get_legend_handles_labels()
ax0.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8)

# --- Panel 2: desecho vs pérdida de área ---
axes[1].scatter(inf["loss_global_pct"], inf["q_pct"], c="#176B87", s=55)
for _, r in inf.iterrows():
    axes[1].annotate(str(int(r["Week"])),
        (r["loss_global_pct"], r["q_pct"]), xytext=(5,5),
        textcoords="offset points", fontsize=8)
axes[1].set(xlabel="Pérdida área (%)", ylabel="Desecho (%)",
            title="Desecho vs pérdida área")

# --- Panel 3: desecho vs V6 (mantiene U de V6, que sí es válida) ---
axes[2].errorbar(inf["V6_V"], inf["q_pct"], xerr=inf["U_V6"],
                 fmt="o", color="#6F4E7C", ecolor="#6F4E7C",
                 capsize=3, ms=6)
for _, r in inf.iterrows():
    axes[2].annotate(str(int(r["Week"])),
        (r["V6_V"], r["q_pct"]), xytext=(5,5),
        textcoords="offset points", fontsize=8)
axes[2].set(xlabel="V6 ± U (V)", ylabel="Desecho (%)", title="Desecho vs V6")

fig.suptitle("Condición óptica vs desempeño productivo", fontsize=14)
fig.tight_layout()

p = FIG_DIR/"10_desecho_vs_opticos.png"
fig.savefig(p, dpi=220); plt.show()

register_figure("Desecho semanal vs condición óptica", p,
    "Izq.: evolución semanal del desecho y la pérdida de área. "
    "Centro: desecho vs pérdida de área. Der.: desecho vs V6.")

## 10. Comparación cruzada ÁREA vs V6

In [ ]:
cross_r2 = (area_cmp
    .set_index("Model")[["R2_adj","q","RMSE"]]
    .rename(columns={"R2_adj":"R2adj_area","q":"q_area","RMSE":"RMSE_area"})
    .join(v6_cmp
        .set_index("Model")[["R2_adj","q","RMSE"]]
        .rename(columns={"R2_adj":"R2adj_v6","q":"q_v6","RMSE":"RMSE_v6"}),
        how="outer"))
cross_r2 = cross_r2.sort_values("R2adj_area", ascending=False)

cross_disp = pd.DataFrame({
    "Modelo": cross_r2.index,
    "q": cross_r2["q_area"].fillna(cross_r2["q_v6"]).astype("Int64"),
    "R²aj ÁREA": cross_r2["R2adj_area"].map(
        lambda v: "—" if pd.isna(v) else f"{v:.4f}"),
    "RMSE ÁREA": cross_r2["RMSE_area"].map(
        lambda v: "—" if pd.isna(v) else f"{v:.5f}"),
    "R²aj V6": cross_r2["R2adj_v6"].map(
        lambda v: "—" if pd.isna(v) else f"{v:.4f}"),
    "RMSE V6 (V)": cross_r2["RMSE_v6"].map(
        lambda v: "—" if pd.isna(v) else f"{v:.3f}"),
})
show_colab_table(cross_disp, "R² ajustado comparado — mismo modelo, dos métricas")
register_table("R² ajustado comparado: ÁREA vs V6", cross_disp,
    "Ganadores independientes por R² ajustado. Gompertz domina ambas métricas en ajuste, pero Seminormal gana en LOO para V6.")

In [ ]:
# --- Gráfico 13 + 14 ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ord_a = area_cmp.sort_values("RMSE")
axes[0].barh(ord_a["Model"], ord_a["RMSE"], color="#176B87")
axes[0].invert_yaxis()
axes[0].set_xlabel("RMSE del área")
axes[0].set_title(f"Ranking del ÁREA — ganador AICc: {winner_area_aicc}")

ord_v = v6_cmp.sort_values("RMSE")
axes[1].barh(ord_v["Model"], ord_v["RMSE"], color="#C8553D")
axes[1].invert_yaxis()
axes[1].set_xlabel("RMSE de V6 (V)")
axes[1].set_title(f"Ranking de V6 — seleccionado: {sel_name}")

fig.suptitle("Comparación: modelo del área vs modelo de V6",
             fontsize=14, fontweight="bold")
fig.tight_layout()
p = FIG_DIR/"13_ranking_cruzado.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Ranking cruzado ÁREA vs V6", p,
    "Izq.: RMSE del área. Der.: RMSE de V6.")

# Integrado 2x2
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
cmap = plt.get_cmap("tab10")
x_dense_a = np.linspace(x_area.min(), x_area.max(), 400)
x_dense_v = np.linspace(x_v6.min(), 4 * x_v6.max(), 400)

axes[0, 0].errorbar(x_area, y_area, yerr=area_md["U_A_norm"],
                    fmt="o", color="black", ecolor="#555", capsize=3,
                    label="Observado ± U", zorder=5)
for i, spec in enumerate(cands):
    axes[0, 0].plot(x_dense_a,
                    spec["f"](fits[spec["Model"]]["res"].x, x_dense_a),
                    lw=1.5, color=cmap(i), label=spec["Model"])
axes[0, 0].set(xlabel="Días desde 30-may", ylabel="Área normalizada",
               title="ÁREA — Ajustes")
axes[0, 0].legend(fontsize=7, ncol=2)

ord_a = area_cmp.sort_values("RMSE")
axes[0, 1].barh(ord_a["Model"], ord_a["RMSE"], color="#176B87")
axes[0, 1].invert_yaxis()
axes[0, 1].set_xlabel("RMSE del área")
axes[0, 1].set_title(f"ÁREA — Ranking (AICc: {winner_area_aicc})")

axes[1, 0].errorbar(x_v6, y_v6, yerr=v6_summary["U_V6"],
                    fmt="o", color="black", ecolor="#555", capsize=3,
                    label="V6 observado ± U", zorder=5)
for i, spec in enumerate(v6_cands):
    axes[1, 0].plot(x_dense_v,
                    spec["f"](v6_fits[spec["Model"]]["res"].x, x_dense_v),
                    lw=1.5, color=cmap(i), label=spec["Model"])
axes[1, 0].axhline(OPERATING_V_MAX, color="#C23B22", ls="--", lw=1.5,
                   label="Límite 190 V")
axes[1, 0].set(xlabel="Días desde 30-may", ylabel="V6 (V)", title="V6 — Ajustes")
axes[1, 0].legend(fontsize=7, ncol=2)

ord_v = v6_cmp.sort_values("RMSE")
axes[1, 1].barh(ord_v["Model"], ord_v["RMSE"], color="#C8553D")
axes[1, 1].invert_yaxis()
axes[1, 1].set_xlabel("RMSE de V6 (V)")
axes[1, 1].set_title(f"V6 — Seleccionado: {sel_name}")

fig.suptitle("Comparación integrada ÁREA vs V6", fontsize=15, fontweight="bold")
fig.tight_layout()
p = FIG_DIR/"14_comparacion_integrada.png"
fig.savefig(p, dpi=220); plt.show()
register_figure("Comparación integrada ÁREA vs V6", p,
    "Panel 2x2 con los ajustes y rankings de ambas métricas.")

## 11. Recomendaciones

In [ ]:
recomendaciones = '''
RECOMENDACIONES PARA EL SEGUIMIENTO:

1. AMPLIAR LA VENTANA DE MEDICIÓN (crítico)
   Con solo 7 sesiones (43 días), el análisis no puede determinar si V6
   saturará por debajo de 190 V o si continuará creciendo por encima. Se
   recomienda extender el seguimiento al menos 2-3 sesiones adicionales
   (agosto, septiembre, octubre). Con 10 puntos en lugar de 7, el análisis
   de modelos será concluyente.

2. VERIFICAR LA SATURACIÓN DE V6
   - Si en las próximas sesiones V6 continúa en 183-185 V → saturación
     confirmada, la lámpara no cruzará 190 V por criterio de 6 W.
   - Si V6 sube a 187 V o más → la saturación previa era un artefacto
     de pocos datos, se requiere reemplazo.

3. REVISAR EL CRITERIO DE REEMPLAZO
   El criterio actual de "V6 > 190 V" puede ser insuficiente. La caída
   de área supera el 51 % en el mismo período. Si el proceso depende de
   la respuesta global de la lámpara, se recomienda definir un segundo
   criterio basado en el área (por ejemplo, pérdida > 40 %).

4. RESOLVER LA AMBIGÜEDAD DE DATOS DE 199 V
   Se identificaron dos mediciones con 199 V en las sesiones 30-may,
   07-jun y 13-jun. El segundo registro se reetiquetó a 200 V por
   coherencia con las sesiones posteriores. Se recomienda verificar
   con la bitácora original de las mediciones.

5. ANALIZAR EL DESECHO CON MÁS VARIABLES DE CONTROL
   La correlación parcial por tiempo sugiere que el desecho está dominado
   por el ramp-up operativo, no por el estado de la lámpara. Se recomienda
   registrar variables adicionales (temperatura de proceso, lote de
   material, turno) para aislar mejor el efecto de la lámpara.

6. DOCUMENTAR LA CADENCIA DE LAS SIGUIENTES MEDICIONES
   Para poder aplicar validación cruzada temporal correctamente, se
   recomienda mantener la cadencia semanal de las mediciones y registrar
   cada sesión con fecha exacta.
'''
print(recomendaciones)
register_text("Recomendaciones para el seguimiento", recomendaciones)

## 12. Generación del reporte

In [ ]:
def fig_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def df_to_html_table(df):
    return df.to_html(index=False, escape=False, classes="data-table", border=0)

def build_html_report(items, title="Análisis de degradación de lámpara"):
    css = '''
    <style>
    body {font-family: Helvetica, Arial, sans-serif; max-width: 1100px;
          margin: 30px auto; padding: 20px; color: #222; line-height: 1.5;}
    h1 {color: #176B87; border-bottom: 3px solid #176B87; padding-bottom: 10px;}
    h2 {color: #176B87; margin-top: 40px; border-left: 5px solid #176B87;
        padding-left: 12px;}
    .meta {color: #666; font-size: 0.9em; margin-bottom: 30px;}
    .data-table {border-collapse: collapse; margin: 12px 0 24px 0;
                 font-size: 13px; width: 100%;}
    .data-table th {background: #176B87; color: white; padding: 8px 10px;
                    text-align: center; border-bottom: 2px solid #0F4C5C;}
    .data-table td {padding: 6px 10px; text-align: right;
                    border-bottom: 1px solid #D9E2E8;}
    .data-table tr:nth-child(even) {background: #F4F8FA;}
    .figure-block {margin: 20px 0 35px 0; text-align: center;}
    .figure-block img {max-width: 100%; height: auto;
                       border: 1px solid #DDD;
                       box-shadow: 0 2px 6px rgba(0,0,0,0.1);}
    .caption {font-size: 0.9em; color: #555; font-style: italic;
              margin-top: 8px;}
    .note {background: #FFF8E1; border-left: 4px solid #F5A623;
           padding: 10px 15px; margin: 10px 0 20px 0; font-size: 0.9em;}
    .warning {background: #FFEBEE; border-left: 4px solid #C23B22;
              padding: 12px 18px; margin: 15px 0 25px 0; font-size: 0.92em;}
    @media print {body {margin: 0;}
                  .figure-block {page-break-inside: avoid;}}
    </style>
    '''
    html = f'<!DOCTYPE html><html><head><meta charset="utf-8"><title>{title}</title>{css}</head><body>'
    html += f'<h1>{title}</h1>'
    html += f'<div class="meta">Reporte generado automáticamente. '
    html += f'{sum(1 for x in items if x["type"]=="table")} tablas y '
    html += f'{sum(1 for x in items if x["type"]=="figure")} figuras.</div>'

    for i, item in enumerate(items, 1):
        if item["type"] == "text":
            html += f'<h2>{item["title"]}</h2>'
            cls = "warning" if "advertencia" in item["title"].lower() or "recomend" in item["title"].lower() else "note"
            html += f'<div class="{cls}"><pre style="white-space:pre-wrap;margin:0;font-family:inherit;">{item["body"]}</pre></div>'
        elif item["type"] == "table":
            html += f'<h2>Tabla {i}. {item["title"]}</h2>'
            if item.get("note"):
                html += f'<div class="note">{item["note"]}</div>'
            html += df_to_html_table(item["df"])
        elif item["type"] == "figure":
            html += f'<h2>Figura {i}. {item["title"]}</h2>'
            html += '<div class="figure-block">'
            b64 = fig_to_base64(item["path"])
            html += f'<img src="data:image/png;base64,{b64}" alt="{item["title"]}">'
            if item.get("caption"):
                html += f'<div class="caption">{item["caption"]}</div>'
            html += '</div>'
    html += '</body></html>'
    return html

html_report = build_html_report(REPORT_ITEMS)
html_path = OUTPUT_DIR / "reporte_completo.html"
html_path.write_text(html_report, encoding="utf-8")
print(f"Reporte HTML: {html_path.resolve()}")
print(f"Tamaño: {len(html_report)/1024:.1f} KB")

In [ ]:
if DOCX_OK:
    doc = Document()
    style = doc.styles["Normal"]
    style.font.name = "Calibri"; style.font.size = Pt(11)

    h1 = doc.add_heading("Análisis de degradación de lámpara", level=0)
    h1.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p = doc.add_paragraph(
        f"Reporte generado automáticamente. "
        f"{sum(1 for x in REPORT_ITEMS if x['type']=='table')} tablas y "
        f"{sum(1 for x in REPORT_ITEMS if x['type']=='figure')} figuras.")
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER

    tcount = fcount = 0
    for item in REPORT_ITEMS:
        if item["type"] == "text":
            doc.add_heading(item["title"], level=1)
            for line in item["body"].split("\n"):
                if line.strip():
                    doc.add_paragraph(line.strip())
        elif item["type"] == "table":
            tcount += 1
            doc.add_heading(f"Tabla {tcount}. {item['title']}", level=2)
            if item.get("note"):
                np_ = doc.add_paragraph()
                r = np_.add_run(item["note"]); r.italic = True; r.font.size = Pt(9)
            df_show = item["df"].iloc[:30, :10]
            t = doc.add_table(rows=1, cols=len(df_show.columns))
            t.style = "Light Grid Accent 1"
            for j, col in enumerate(df_show.columns):
                t.rows[0].cells[j].text = str(col)
            for _, row in df_show.iterrows():
                cells_row = t.add_row().cells
                for j, val in enumerate(row):
                    cells_row[j].text = str(val)
        elif item["type"] == "figure":
            fcount += 1
            doc.add_heading(f"Figura {fcount}. {item['title']}", level=2)
            try:
                doc.add_picture(item["path"], width=Inches(6.0))
                doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
                if item.get("caption"):
                    cap = doc.add_paragraph()
                    cap_run = cap.add_run(item["caption"])
                    cap_run.italic = True; cap_run.font.size = Pt(9)
                    cap.alignment = WD_ALIGN_PARAGRAPH.CENTER
            except Exception as e:
                doc.add_paragraph(f"[Error imagen: {e}]")

    docx_path = OUTPUT_DIR / "reporte_completo.docx"
    doc.save(str(docx_path))
    print(f"Reporte DOCX: {docx_path.resolve()}")
    print(f"  {tcount} tablas, {fcount} figuras")

In [ ]:
# --- Exportación final ---
area_summary.to_csv(OUTPUT_DIR/"area_160_190V.csv", index=False)
nine_summary.to_csv(OUTPUT_DIR/"indice_R9.csv", index=False)
v6_summary.to_csv(OUTPUT_DIR/"v6_sesiones.csv", index=False)
area_cmp.to_csv(OUTPUT_DIR/"comparacion_modelos_area.csv", index=False)
v6_cmp.to_csv(OUTPUT_DIR/"comparacion_modelos_v6.csv", index=False)
wa.to_csv(OUTPUT_DIR/"desecho_vs_opticos.csv", index=False)
corr_df.to_csv(OUTPUT_DIR/"correlaciones.csv", index=False)

archive = Path(shutil.make_archive("resultados_lampara","zip", root_dir=OUTPUT_DIR))
print("\nZIP:", archive.resolve())
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file(): print(f"  - {p.name}")
    elif p.is_dir():
        print(f"  - {p.name}/")
        for f in sorted(p.iterdir()): print(f"      {f.name}")

from google.colab import files
files.download(str(archive))
# files.download(str(OUTPUT_DIR/"reporte_completo.docx"))
print("\nTodo descargado.")